# Week 6 补充：正确的模型选择与最终测试

目标：建立训练集、交叉验证和最终测试集的正确边界。测试集从一开始保留，不参与模型选择或参数调整。

In [1]:
import pandas as pd

from sklearn.datasets import load_breast_cancer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.tree import DecisionTreeClassifier

## 1. 第一步先保留最终测试集

这一步只做一次。后续所有交叉验证、模型选择和调参都只使用 `X_train` 与 `y_train`。

In [2]:
dataset = load_breast_cancer(as_frame=True)
X = dataset.data
y = dataset.target

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y,
)

print('train shape:', X_train.shape)
print('test shape:', X_test.shape)
print('train positive ratio:', round(y_train.mean(), 3))
print('test positive ratio:', round(y_test.mean(), 3))

train shape: (455, 30)
test shape: (114, 30)
train positive ratio: 0.626
test positive ratio: 0.632


## 2. 只在训练集内做 5 折交叉验证

这里的每一个验证折都来自训练集。`X_test` 和 `y_test` 在此单元中完全不出现。

In [3]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)

models = {
    'decision tree (max_depth=3)': DecisionTreeClassifier(max_depth=3, random_state=42),
    'random forest': RandomForestClassifier(
        n_estimators=300,
        max_features='sqrt',
        random_state=42,
        n_jobs=-1,
    ),
}

rows = []
for name, model in models.items():
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring='accuracy', n_jobs=-1)
    rows.append({
        'model': name,
        'cv_accuracy_mean': scores['test_score'].mean(),
        'cv_accuracy_std': scores['test_score'].std(),
    })

cv_results = pd.DataFrame(rows).sort_values('cv_accuracy_mean', ascending=False)
display(cv_results)

,model,cv_accuracy_mean,cv_accuracy_std
1,random forest,0.964835,0.017582
0,decision tree (max_depth=3),0.925275,0.008223


## 3. 选定模型后，用完整训练集重训一次，再测试一次

本例根据交叉验证选择随机森林。此时才允许使用 `X_test` 和 `y_test`。不要根据这里的结果回头改参数；否则测试集也会被用于模型选择。

In [4]:
final_model = RandomForestClassifier(
    n_estimators=300,
    max_features='sqrt',
    random_state=42,
    n_jobs=-1,
)

final_model.fit(X_train, y_train)
test_accuracy = accuracy_score(y_test, final_model.predict(X_test))

print('final test accuracy:', round(test_accuracy, 3))
print('This number is for final reporting, not for further tuning.')

final test accuracy: 0.947
This number is for final reporting, not for further tuning.


## 检查点

1. 为什么 `X_test` 不能放进交叉验证？
2. 为什么最终模型要在完整 `X_train` 上重新 `fit` 一次？
3. 如果最终测试集表现不好，为什么不能反复根据它调参？